In [1]:
import re
import math
import random
from collections import defaultdict, Counter
from tqdm import tqdm

In [2]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)   # bỏ dấu câu
    text = re.sub(r'\d+', '', text)        # bỏ số
    text = re.sub(r'\s+', ' ', text).strip()
    return text
# Load data
with open('data/truyen_kieu_data.txt', 'r', encoding='utf-8') as f:
    raw_text = f.read()
cleaned = preprocess(raw_text)
tokens = cleaned.split()
print(f"Tổng số token: {len(tokens)}")
print("Ví dụ:", tokens[:20])

Tổng số token: 22806
Ví dụ: ['trăm', 'năm', 'trong', 'cõi', 'người', 'ta', 'chữ', 'tài', 'chữ', 'mệnh', 'khéo', 'là', 'ghét', 'nhau', 'trải', 'qua', 'một', 'cuộc', 'bể', 'dâu']


In [3]:
split_idx = int(0.8 * len(tokens))
train_tokens = tokens[:split_idx]
test_tokens  = tokens[split_idx:]
print(f"Train: {len(train_tokens)} tokens")
print(f"Test : {len(test_tokens)}  tokens")

Train: 18244 tokens
Test : 4562  tokens


In [4]:
class NGramModel:
    def __init__(self, n=2):
        self.n = n
        self.ngram_counts  = defaultdict(Counter)  # context → {next_word: count}
        self.context_totals = defaultdict(int)      # context → tổng số lần xuất hiện
        self.vocab = set()

    def train(self, tokens):
        self.vocab = set(tokens)
        V = len(self.vocab)
        total_ngrams = len(tokens) - self.n + 1

        print("=" * 55)
        print(f"  🚀 Bắt đầu training N-gram Model (N={self.n})")
        print("=" * 55)
        print(f"  📄 Tổng số token        : {len(tokens):,}")
        print(f"  📚 Kích thước Vocabulary : {V:,} từ")
        print(f"  🔢 Tổng số N-gram        : {total_ngrams:,}")
        print()

    # ── Vòng lặp đếm n-gram có progress bar ────────────────
        for i in tqdm(range(total_ngrams),
                    desc=f"  Counting {self.n}-grams",
                    unit=" ngram",
                    ncols=70,
                    colour="green"):
            ngram   = tuple(tokens[i : i + self.n])
            context = ngram[:-1]
            word    = ngram[-1]
            self.ngram_counts[context][word] += 1
            self.context_totals[context]     += 1

        self.V = V

    # ── Thống kê sau khi xong ───────────────────────────────
        print(f"\n  🗂️  Số context (prefix)   : {len(self.ngram_counts):,}")

        top_ctx = sorted(self.context_totals.items(),
                         key=lambda x: x[1], reverse=True)[:5]
        print("\n  📊 Top 5 context xuất hiện nhiều nhất:")
        for ctx, cnt in top_ctx:
            top_next = self.ngram_counts[ctx].most_common(3)
            next_str = ", ".join([f"'{w}'({c})" for w, c in top_next])
            print(f"     {ctx}  →  [{next_str}]  (tổng: {cnt})")

        print("\n" + "=" * 55)
        print("  ✅ Training hoàn tất!")
        print("=" * 55)


    def prob(self, context, word):
        context = tuple(context)
        count_w = self.ngram_counts[context].get(word, 0)
        count_c = self.context_totals.get(context, 0)
        # Laplace (add-1) smoothing
        return (count_w + 1) / (count_c + self.V)

    # Dự đoán từ tiếp theo 
    def predict_next(self, context, top_k=5):
        """Trả về top_k từ có xác suất cao nhất."""
        context = tuple(context[-(self.n - 1):])   # lấy n-1 từ cuối
        candidates = self.ngram_counts.get(context, {})

        if not candidates:
            return []

        # Tính xác suất cho tất cả ứng viên
        scored = {w: self.prob(context, w) for w in candidates}
        return sorted(scored.items(), key=lambda x: x[1], reverse=True)[:top_k]


In [5]:
N = 3          # thử Trigram; đổi thành 2 cho Bigram

model = NGramModel(n=N)
model.train(train_tokens)

  🚀 Bắt đầu training N-gram Model (N=3)
  📄 Tổng số token        : 18,244
  📚 Kích thước Vocabulary : 2,284 từ
  🔢 Tổng số N-gram        : 18,242



  Counting 3-grams:  93%|▉| 16890/18242 [00:00<00:00, 142150.70 ngram/

  Counting 3-grams: 100%|█| 18242/18242 [00:00<00:00, 143167.34 ngram/


  🗂️  Số context (prefix)   : 14,907

  📊 Top 5 context xuất hiện nhiều nhất:
     ('nàng', 'rằng')  →  ['trời'(2), 'muôn'(2), 'này'(1)]  (tổng: 28)
     ('một', 'lời')  →  ['đã'(2), 'là'(1), 'nói'(1)]  (tổng: 19)
     ('nàng', 'mới')  →  ['giãi'(3), 'thưa'(2), 'theo'(2)]  (tổng: 19)
     ('tiểu', 'thư')  →  ['lại'(2), 'phải'(2), 'nổi'(1)]  (tổng: 19)
     ('một', 'mình')  →  ['cho'(2), 'lặng'(1), 'thiu'(1)]  (tổng: 17)

  ✅ Training hoàn tất!


In [6]:
def compute_perplexity(model, tokens):
    n       = model.n
    log_sum = 0.0
    count   = 0

    for i in range(n - 1, len(tokens)):
        context = tuple(tokens[i - (n - 1) : i])
        word    = tokens[i]
        p       = model.prob(context, word)
        log_sum += math.log2(p)
        count   += 1
    H_W = -log_sum / count     

    return math.pow(2, H_W)

pp = compute_perplexity(model, test_tokens)
print(f"Perplexity trên tập test (N={N}): {pp:.2f}")

Perplexity trên tập test (N=3): 2231.35


In [7]:
def generate_sentence(model, seed_words, max_words=30):
    """
    seed_words : list từ ban đầu, ví dụ ['trăm', 'năm']
    Trả về câu hoàn chỉnh dưới dạng string.
    """
    words = list(seed_words)
    n     = model.n

    for _ in range(max_words):
        context    = tuple(words[-(n - 1):])
        candidates = model.predict_next(context, top_k=10)

        if not candidates:
            break   # không tìm được context phù hợp → dừng

        # Lấy ngẫu nhiên có trọng số từ top-k
        next_words  = [w for w, _ in candidates]
        probs       = [p for _, p in candidates]
        total       = sum(probs)
        norm_probs  = [p / total for p in probs]

        next_word = random.choices(next_words, weights=norm_probs, k=1)[0]
        words.append(next_word)

    return ' '.join(words)

# ── Thử sinh câu ──────────────────────────────────────────────────
seed = ['trăm', 'năm']
sentence = generate_sentence(model, seed_words=seed, max_words=25)
print("Seed:", seed)
print("Câu sinh ra:", sentence)


Seed: ['trăm', 'năm']
Câu sinh ra: trăm năm tạc một chữ đồng tâm trăm năm tính cuộc vuông tròn cho chăng ngần ngừ nàng mới lựa dây nỉ non thánh thót dễ say


In [8]:
for n in [2, 3, 4]:
    m = NGramModel(n=n)
    m.train(train_tokens)
    pp = compute_perplexity(m, test_tokens)
    print(f"N={n}  → Perplexity: {pp:.2f}")


  🚀 Bắt đầu training N-gram Model (N=2)
  📄 Tổng số token        : 18,244
  📚 Kích thước Vocabulary : 2,284 từ
  🔢 Tổng số N-gram        : 18,243



  Counting 2-grams: 100%|█| 18243/18243 [00:00<00:00, 246119.04 ngram/



  🗂️  Số context (prefix)   : 2,284

  📊 Top 5 context xuất hiện nhiều nhất:
     ('một',)  →  ['lời'(19), 'mình'(17), 'ngày'(13)]  (tổng: 260)
     ('đã',)  →  ['thấy'(9), 'đến'(6), 'biết'(5)]  (tổng: 208)
     ('người',)  →  ['ta'(5), 'đâu'(5), 'biết'(4)]  (tổng: 180)
     ('nàng',)  →  ['rằng'(28), 'mới'(19), 'đã'(9)]  (tổng: 160)
     ('cho',)  →  ['người'(9), 'ai'(6), 'hay'(5)]  (tổng: 149)

  ✅ Training hoàn tất!
N=2  → Perplexity: 1638.12
  🚀 Bắt đầu training N-gram Model (N=3)
  📄 Tổng số token        : 18,244
  📚 Kích thước Vocabulary : 2,284 từ
  🔢 Tổng số N-gram        : 18,242



  Counting 3-grams: 100%|█| 18242/18242 [00:00<00:00, 152315.50 ngram/



  🗂️  Số context (prefix)   : 14,907

  📊 Top 5 context xuất hiện nhiều nhất:
     ('nàng', 'rằng')  →  ['trời'(2), 'muôn'(2), 'này'(1)]  (tổng: 28)
     ('một', 'lời')  →  ['đã'(2), 'là'(1), 'nói'(1)]  (tổng: 19)
     ('nàng', 'mới')  →  ['giãi'(3), 'thưa'(2), 'theo'(2)]  (tổng: 19)
     ('tiểu', 'thư')  →  ['lại'(2), 'phải'(2), 'nổi'(1)]  (tổng: 19)
     ('một', 'mình')  →  ['cho'(2), 'lặng'(1), 'thiu'(1)]  (tổng: 17)

  ✅ Training hoàn tất!
N=3  → Perplexity: 2231.35
  🚀 Bắt đầu training N-gram Model (N=4)
  📄 Tổng số token        : 18,244
  📚 Kích thước Vocabulary : 2,284 từ
  🔢 Tổng số N-gram        : 18,241



  Counting 4-grams: 100%|█| 18241/18241 [00:00<00:00, 148206.41 ngram/



  🗂️  Số context (prefix)   : 17,981

  📊 Top 5 context xuất hiện nhiều nhất:
     ('lời', 'nàng', 'mới')  →  ['bước'(1), 'theo'(1), 'lựa'(1)]  (tổng: 5)
     ('rằng', 'chút', 'phận')  →  ['ngây'(2), 'bọt'(1), 'lạc'(1)]  (tổng: 4)
     ('riêng', 'riêng', 'những')  →  ['bàng'(1), 'sụt'(1), 'nặng'(1)]  (tổng: 4)
     ('đến', 'thế', 'này')  →  ['thì'(2), 'chẳng'(1), 'thôi'(1)]  (tổng: 4)
     ('thế', 'thì', 'thôi')  →  ['đời'(1), 'còn'(1), 'rằng'(1)]  (tổng: 3)

  ✅ Training hoàn tất!
N=4  → Perplexity: 2272.75
